In [ ]:
import os
import requests
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.tools import tool
from langchain_ollama import ChatOllama

load_dotenv()
SERPER_API_KEY = os.environ["SERPER_API_KEY"]


@tool
def search_linkedin_profiles(query: str) -> str:
    """Search Google for public LinkedIn profiles.
    Pass a query like 'Senior Machine Learning Engineer Amman Jordan'."""
    response = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"},
        # 2. In the tool: fewer results, and always pin the location to Amman
        json={"q": f"site:linkedin.com/in {query} Amman", "num": 6, "gl": "jo"},
        timeout=15,
    )
    response.raise_for_status()
    results = response.json().get("organic", [])
    if not results:
        return "No profiles found."
    return "\n".join(
        f"- {r.get('title')}\n  {r.get('snippet')}\n  {r.get('link')}" for r in results
    )


# 1. Give the model a bigger context window
model = ChatOllama(model="gpt-oss:20b", num_ctx=16384)

agent = create_agent(
    model=model,
    tools=[search_linkedin_profiles],
    system_prompt=(
    "You are a recruiting assistant. Find the strongest AI professionals in Jordan, "
    "such as machine learning engineers, AI engineers, data scientists and AI researchers. "
    "Jordan is the COUNTRY, not a person's name. "
    "Try a few different searches (e.g. 'Senior Machine Learning Engineer', "
    "'AI Engineer LLM', 'Data Scientist Deep Learning'), but do at most 3 searches, "
    "then write your answer. "
    "Rank the candidates by seniority, years of experience, and signs of strong AI work "
    "shown in their snippets, like leading AI teams, research, publications or shipped AI products. "
    "Give each one's name, title, a short reason, and the profile link. "
    "Only use what the search results show; never invent details."
    ),
    middleware=[
        ToolCallLimitMiddleware(   #
            tool_name="search_linkedin_profiles",  # which tool to limit
            run_limit=3,                           # max 3 searches per run
            exit_behavior="continue",              # block extra calls, let the model answer
        ),
    ],
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Find the top 5 strongest AI professionals in Jordan."}]
})
print(result["messages"][-1].content)
# print(result)

**Top 5 AI Professionals in Jordan (ranked by seniority, experience, and demonstrable AI impact)**  

| Rank | Name | Current Title | Why They Rank High | LinkedIn |
|------|------|---------------|--------------------|----------|
| 1 | **Mutasem AlDarabsah** | Senior Applied Scientist | Long‑term experience in high‑scale ML at leading e‑commerce and tech firms (Souq.com, Samsung). Deep‑learning expertise across multiple product lines, proving ability to ship production‑ready AI systems. | <https://www.linkedin.com/in/mutasem-mattar> |
| 2 | **Bara’ Al‑Etawi** | Senior AI Engineer (Generative AI & LLM) | Specialized in building LLM‑based, production‑ready solutions. Demonstrated leadership in delivering end‑to‑end GenAI pipelines that have moved to production at Jordanian tech startups. | <https://jo.linkedin.com/in/baraaetawi> |
| 3 | **Ahmad Hammad** | Senior AI Engineer | 4‑year AWS certification track record and hands‑on experience in MLOps, machine‑learning, and AI product delivery